# Tutorial 4. Bayesian statistics in configuration search


## Goal of the notebook

The goal of this example is to demonstrate the use of the BEACON as implemented in GPAtom and ASE to a 13-atom Ag-Pd-Pt-Ru nanoparticle.

## Description of the method

The BEACON method applies Bayesian optimization to efficiently search complex configurational spaces in multimetallic systems. Atomic structures are represented using global structural fingerprints that encode both composition and geometry, and their energies are modeled with a Gaussian process surrogate model. During the search, new candidate configurations are generated and evaluated, while the surrogate model guides the exploration toward low-energy structures. This approach enables efficient identification of stable atomic arrangements in alloy nanoparticles, where compositional ordering and atomic positions strongly influence catalytic properties.

BEACON gitlab: https://gitlab.com/gpatom/ase-gpatom

BEACON article: Kaappa, S.; del Río, E. G.; Jacobsen, K. W. Global Optimization of Atomic Structures with Gradient-Enhanced Gaussian Process Regression. Phys. Rev. B 2021, 103 (17), 174114. https://doi.org/10.1103/PhysRevB.103.174114
.
## Outline of the tutorial

STEP 1. Nanoparticle search

STEP 2. Place OH on the lowest energy  nanoparticle

STEP 3. Best adsorption location search

## Installation of packages


First thing first, we need to install the packages and import the neccessary functions.



Next we import the necessary packages:

ASE https://ase-lib.org/

TBLite https://github.com/tblite

GPAtom https://gitlab.com/gpatom


In [ ]:
#############
##-Imports-##
#############

##-ASE
from ase import Atoms, Atom
from ase.io import read, write
from ase.visualize import view
from ase.data import covalent_radii, atomic_numbers
from ase.constraints import FixAtoms, FixBondLengths

##-TBLite
from tblite.ase import TBLite

##-GPAtom
from gpatom.gpfp.prior import RepulsivePotential, CalculatorPrior
from gpatom.beacon.str_gen import  RandomBox, AtomsRelaxer, RandomMolecule, RandomStructure
from gpatom.gpfp.gp import GaussianProcess
from gpatom.gpfp.fingerprint import FingerPrint, FPUpdater
from gpatom.gpfp.avishart_hpfitter import HpFitterConstantRatioParallel
from gpatom.gpfp.atoms_gp_interface import LCBModel
from gpatom.beacon.beacon import  (BEACON, SurrogateOptimizer,InitatomsGenerator, Checker)

##-Other
import numpy as np


## STEP 1. Nanoparticle Search

BEACON generates random atomic arrangements of the alloy and evaluates their energies using TBLite calculator. This energy is inturn used by a Gaussian-process model surrogate which guides the search toward promising structures, essentially locating the low-energy geometries.

In [ ]:
########################
##-Creating the alloy-##
########################

##-Creating the alloy
elements = Atoms(['Pt']*3 + ['Pd']*2 + ['Ag']*3 + ['Ru']*5,  cell = [15,15,15], pbc=False)
elements.calc = TBLite(method="GFN1-xTB", verbosity = 0)

In [ ]:
################################
##-Structure generation setup-##
################################

##-First we define a repulsive potential to make sure that atoms arent placed ontop  of each other, this is a simple lennard jones ase calculator
potential=RepulsivePotential(prefactor=10, rc=0.9)

##-Then a relaxtion algoritm using the potential as calculator, This is an interface that call BFGS, it might be replaceable with a normal optimisers
relaxer=AtomsRelaxer(calculator=potential, with_unit_cell=False)

##-The random generator, we dont have multple ranks so no need to account for it in this instance
rng = np.random.RandomState(42)

system_gen = RandomBox(elements, box=[(5, 10)]*3, rng=rng, relaxer=relaxer)
initatomsgen=InitatomsGenerator(sgen=system_gen, rgen=system_gen)

##-Now we have all the code for generating structures

In [ ]:
###########################
##-Surrogate model setup-##
###########################

##-Sets and interfaces the calculator used in fingerprinting
prior_calculator = potential ##-TBLite(method="GFN1-xTB", verbosity = 0)
prior=CalculatorPrior(potential, constant=0)

gp_args = dict(prior=prior,
               hp={'scale': 1000, 'weight': 100,
                   'ratio': 0.001, 'noisefactor': 1},
               use_forces=True)

gp=GaussianProcess(**gp_args)


##-Next we set the size of a Atom fingerprint to  that of the largest atom
r_atom = max(covalent_radii[atomic_numbers['Pt']],
             covalent_radii[atomic_numbers['Pd']],
             covalent_radii[atomic_numbers['Ag']],
             covalent_radii[atomic_numbers['Ru']])
fp_args = {'r_cutoff': r_atom*5, 'a_cutoff': r_atom*3, 'aweight': 1}
fp=FingerPrint(fp_args=fp_args, calc_strain=False)

##-Optimiser for parameters
hp_optimizer = HpFitterConstantRatioParallel()

##-Fingerprint updater
fp_updater=FPUpdater(factor=1/3)

##-Finally everything is put together in the ML model
model=LCBModel(gp=gp, fp=fp, hp_optimizer=hp_optimizer, fp_updater=fp_updater, kappa=2)

In [ ]:
###################
##-BEACON search-##
###################

surropt=SurrogateOptimizer(fmax=0.05, relax_steps=100,
                           with_unit_cell=False)

##-The checker check structure likeness
checker=Checker(dist_limit=0.5, rlimit=0.4)

calculator = TBLite(method="GFN1-xTB", verbosity = 0)

go = BEACON(calculator,
            model,
            initatomsgen,
            surropt=surropt,
            checker=checker,
            ninit=2, ##-How many training points we want in initial set
            ndft=10, ##-How many TBlite calculations are done in total
            nsur=3, ##-How many surrogate optimizations are done per cycle
            write_surropt=True, ##-Save surrogate relaxation end structures
            write_surropt_trajs=False) ##-Dont save trajectory files for all surrogate relaxations
##-Run:
go.run()

3
{'bond-orders': array([[0.        , 0.02196551, 0.87109488, 0.02623848, 0.04291761,
        0.10291546, 0.28283987, 0.0303406 , 1.08402684, 0.1155192 ,
        0.05058464, 1.87396245, 0.10683218],
       [0.02196551, 0.        , 0.02548641, 0.03955922, 0.15200233,
        0.02184076, 0.04038508, 0.45511062, 0.10934281, 0.39843813,
        1.64980293, 0.04920926, 0.36675129],
       [0.87109488, 0.02548641, 0.        , 0.02611839, 0.05188133,
        0.57361977, 0.06691242, 0.06798722, 1.34033553, 0.19786753,
        0.06387583, 0.4177385 , 0.11210365],
       [0.02623848, 0.03955922, 0.02611839, 0.        , 0.21660571,
        0.02103424, 0.48691607, 0.02125285, 0.19885506, 0.03870919,
        0.08332773, 0.04650291, 0.76670581],
       [0.04291761, 0.15200233, 0.05188133, 0.21660571, 0.        ,
        0.01840188, 0.44851375, 0.05370465, 0.22664203, 0.11401016,
        0.4801596 , 0.10170328, 0.17361715],
       [0.10291546, 0.02184076, 0.57361977, 0.02103424, 0.01840188,
        0

## STEP 2. Placing adsorbate

After the search completes, we choose the lowest-energy nanoparticle structure identified by BEACON. Onto that nanoparticle, we generate an OH molecule, positioned near the nanoparticle while avoiding overlapping. Nanoparticles atom positions are fixed.

In [ ]:
####################
##-Best candidate-##
####################

best_candidate = go.get_candidate()[0]
write('example_4_BEACON_best_result.xyz', best_candidate)

In [ ]:
#################
##-Definitions-##
#################

def sph2cart(r, theta, phi):
    """Convert spherical to Cartesian coordinates."""
    x = r * np.sin(theta) * np.cos(phi)
    y = r * np.sin(theta) * np.sin(phi)
    z = r * np.cos(theta)
    return x, y, z

def cart2sph(x, y, z):
    """Convert Cartesian to spherical coordinates."""
    r = np.sqrt(x**2 + y**2 + z**2)
    theta = np.arccos(z / r)             ##-Polar angle
    phi = np.arctan2(y, x)               ##-azimuth
    return r, theta, phi



def cell_center_cartesian(atoms):
    """Return the Cartesian coordinates of the geometric center of the unit cell."""
    cell = atoms.get_cell()          ##-shape (3,3), cell vectors as rows
    frac_center = np.array([0.5, 0.5, 0.5])
    ##-fractional -> cartesian
    cart_center = frac_center.dot(cell)
    return cart_center


class OHGenerator:
    def __init__(self, rng, relaxer=None):
        self.rng=rng

    def get(self):

        atoms = Atoms('OH', positions=[[0,0,0], [0,0,0.96]])
        atoms.constraints.append(FixBondLengths([[0,1]]))

        for axis in ['x', 'y', 'z']:
            angle = self.rng.uniform(0, 360)
            atoms.rotate(angle, axis, rotate_cell=False)
        return atoms


class MoleculeAroundCenter():
    """
    Generator for randomly placing molecules a structure at the center of the box.
    the generator doesn't change the molecule itself

    this functions is a modified copy of MoleculeOnSubstrate

    Examples:
    >>> sgen=MoleculeOnSubstrate(molecule_generator, adsorbate, dx=2)
    >>> new_atoms=sgen.get()
    """
    def __init__(self, molecule_generator, substrate,  dr=3,
                 relaxer=None, **kwargs):

        """
        Parameters
        ----------
        molecule_generator: BEACON structure generator object
            A structure generator with a method molecule=molecule_generator.get()
            taking no inputs and outputting a molecule subject to custom
            generation logic, including setting of molecular constraints.
            MoleculeOnSubstrate knows how to handle molecules constrained by
            FixAtoms and FixBondLengths
            Required.

        substrate : ase.Atoms
            The substrate structure on which the adsorbate will be placed.
            Required.

        dr : float, optional
            Displacement interval in which the molecule can be placed from
            the center.
            Default is 3 Angstrom.

        relaxer : Relaxer object, optional
            Relaxation procedure applied after structure generation.
            Default is None. (No relaxation)

        **kwargs
            Additional keyword arguments passed to RandomStructure.
            May include:

            rng : random number generator, optional
                Random number generator for reproducibility.
                Default is `np.random` (not reproducible).
        """


        RandomStructure.__init__(self, substrate, **kwargs)
        self.molecule_generator=molecule_generator
        self.substrate=substrate
        self.dr=dr
        self.relaxer=relaxer

    def get(self):

        mol=self.molecule_generator.get()

        substrate=self.substrate.copy()
        mol.cell=substrate.cell.copy()

        ##-get the center of the box in both cartesian and spherical coordinates
        cell=substrate.get_cell()
        cell_center=cell_center_cartesian(mol)
        cell_center_sph=cart2sph(*cell_center)

        ##-this will rely on the mol com not being origo for getting the oriantation of the molecule
        mol_center = mol.get_center_of_mass()
        mol_center_sph = cart2sph(*mol_center)
        mol_center_uv = mol_center/np.linalg.norm(mol_center)

        radia_shift = self.rng.random() * self.dr + 1.05*self.get_minimum_distance(mol_center_uv, np.array(cell_center))
        random_shift = sph2cart(radia_shift,mol_center_sph[1],mol_center_sph[2])

        ##-moving the molecule to the center of the substrate cell
        mol.translate(cell_center)
        ##-moving the molecule in accordance with the random shift
        mol.translate(random_shift)

        atoms=substrate+mol
        atoms.pbc=[True, True, True]

        offset = len(substrate)

        constraints = []
        for c in substrate.constraints:
            constraints.append(c)

        for c in mol.constraints:
            if isinstance(c, FixAtoms):
                new_indices = [i + offset for i in c.index]
                constraints.append(FixAtoms(indices=new_indices))
            elif isinstance(c, FixBondLengths):
                new_bonds = [[i + offset, j + offset] for (i, j) in c.pairs]
                constraints.append(FixBondLengths(new_bonds))

        atoms.set_constraint(constraints)

        if self.relaxer is not None:
            atoms=self.relaxer.run(atoms)

        return atoms


    def get_minimum_distance(self, O_center_unit_vector, center) -> float:
        min_distance = 0
        for at in self.substrate:
            distance_from_at_O_vector = (center - at.position) - np.dot(np.dot((center - at.position), O_center_unit_vector), O_center_unit_vector)
            collision_size = covalent_radii[atomic_numbers['O']] + covalent_radii[atomic_numbers[at.symbol]]
            if np.linalg.norm(distance_from_at_O_vector) < collision_size:
                collision_distance_from_center = np.linalg.norm(np.dot(np.dot((center - at.position), O_center_unit_vector), O_center_unit_vector)) +  np.sqrt(collision_size**2-np.linalg.norm(distance_from_at_O_vector)**2)
                if collision_distance_from_center > min_distance: min_distance = collision_distance_from_center
            else: continue
        return min_distance



In [ ]:
########################
##-OH placement setup-##
########################

##-Now lets ud try to add OH
##-First we will fix all the atoms we dont want the generation to move about
nano_catalyst = best_candidate.copy()
nano_catalyst.set_constraint(FixAtoms(indices=[at.index for at in nano_catalyst]))

##-Next add the OH to the system
##-OH = Atoms('OH', positions=[[0,0,0], [0,0,0.96]])
##-Nano_catalyst += OH

##-First we define a repulsive potential to make sure that atoms arent placed ontop  of each other, this is a simple lennard jones ase calculator
potential=RepulsivePotential(prefactor=10, rc=0.9)

##-Then a relaxtion algoritm using the potential as calculator, This is an interface that call BFGS, it might be replaceable with a normal optimisers
relaxer=AtomsRelaxer(calculator=potential, with_unit_cell=False)

##-The random generator, we dont have multple ranks so no need to account for it in this instance
rng = np.random.RandomState(42)

system_gen = MoleculeAroundCenter(OHGenerator(rng=rng), nano_catalyst, dr=5, rng=rng, relaxer=relaxer)
initatomsgen=InitatomsGenerator(sgen=system_gen, rgen=system_gen)

##-Now we have all the code for generating structures

In [ ]:
print(nano_catalyst)
view(nano_catalyst, viewer='x3d')

Atoms(symbols='Pt3Pd2Ag3Ru5', pbc=False, cell=[15.0, 15.0, 15.0], constraint=FixAtoms(indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]))


## STEP 3. Adsorbate

We use BEACON again to explore different OH placements around the nanoparticle. The model finds the most stable OH adsorption configuration. When the lowest energy nanoparticle is found, we save the best result.


In [ ]:
#############################
##-Adsorption search setup-##
#############################

##-Set up a new model same as before
##-Sets and interfaces the calculator used in fingerprinting
prior_calculator = potential #TBLite(method="GFN1-xTB", verbosity = 0)
prior=CalculatorPrior(potential, constant=0)

gp_args = dict(prior=prior,
               hp={'scale': 1000, 'weight': 100,
                   'ratio': 0.001, 'noisefactor': 1},
               use_forces=True)

gp=GaussianProcess(**gp_args)


##-Next we set the size of a Atom fingerprint to  that of the largest atom
##-We will keep this as is since the size of the metals still matter and are so much larger than the adsorbate radii we are trying to add
r_atom = max(covalent_radii[atomic_numbers['Pt']],
             covalent_radii[atomic_numbers['Pd']],
             covalent_radii[atomic_numbers['Ag']],
             covalent_radii[atomic_numbers['Ru']])
fp_args = {'r_cutoff': r_atom*5, 'a_cutoff': r_atom*3, 'aweight': 1}
fp=FingerPrint(fp_args=fp_args, calc_strain=False)

##-Optimiser for parameters
hp_optimizer = HpFitterConstantRatioParallel()

##-Fingerprint updater
fp_updater=FPUpdater(factor=1/3)

##-Finally everything is put together in the ML model
model=LCBModel(gp=gp, fp=fp, hp_optimizer=hp_optimizer, fp_updater=fp_updater, kappa=2)


In [ ]:
###################
##-BEACON search-##
###################

##-Set up a new beacon same as before.
surropt=SurrogateOptimizer(fmax=0.05, relax_steps=100,
                           with_unit_cell=False)

##-The checker check structure likeness
checker=Checker(dist_limit=0.5, rlimit=0.4)


calculator = TBLite(method="GFN1-xTB", verbosity = 0, max_iterations = 3000)

go = BEACON(calculator,
            model,
            initatomsgen,
            surropt=surropt,
            checker=checker,
            ninit=2, ##-How many training points we want in initial set
            ndft=10, ##-How many TBlite calculations are done in total
            nsur=3, ##-How many surrogate optimizations are done per cycle
            write_surropt=True, ##-Save surrogate relaxation end structures
            write_surropt_trajs=False) ##-Dont save trajectory files for all surrogate relaxations
##-Run:
go.run()

3
{'bond-orders': array([[0.00000000e+00, 1.09045002e+00, 4.58297946e-02, 1.99484418e-01,
        3.11025064e-01, 2.95271891e-02, 6.84922296e-02, 2.80857584e-02,
        2.96189438e-01, 6.17116484e-02, 2.83629255e-01, 1.37074147e+00,
        1.05734717e-01, 5.63538951e-07, 6.31089798e-08],
       [1.09045002e+00, 0.00000000e+00, 4.55126324e-02, 2.02027476e-01,
        1.79764884e-01, 2.95465924e-02, 8.38107050e-02, 9.09164892e-02,
        3.51049315e-01, 4.53053435e-01, 1.48161255e+00, 1.07176000e+00,
        8.26517530e-02, 4.77025840e-07, 4.50422024e-08],
       [4.58297946e-02, 4.55126324e-02, 0.00000000e+00, 1.01092133e-01,
        2.70213080e-01, 4.81125631e-01, 6.23006091e-02, 1.14505416e-01,
        1.22864163e+00, 9.62193152e-01, 2.24453748e-01, 1.71206940e-01,
        1.16762097e+00, 1.00744222e-06, 1.26878086e-07],
       [1.99484418e-01, 2.02027476e-01, 1.01092133e-01, 0.00000000e+00,
        2.83611213e-01, 5.04153939e-02, 1.78494004e-02, 2.02649203e-02,
        6.71848843e

In [ ]:
#######################
##-Best OH candidate-##
#######################

best_OH_candidate = go.get_candidate()[0]
write('example_4_BEACON_best_OH_result.xyz', best_OH_candidate)
print(best_OH_candidate)
view(best_OH_candidate, viewer='x3d')

## Concluding remarks

In this tutorial, we explored how Bayesian optimization can be used to search complex atomic configurations in multimetallic nanoparticles. We began by generating random structures of a 13-atom Ag₃Pd₂Pt₃Ru₅ nanoparticle and used BEACON together with TBLite to identify the lowest-energy arrangement of atoms. Once the most stable nanoparticle configuration was found, an OH molecule was introduced and the search was repeated to determine the most favorable adsorption configuration.

This workflow illustrates how Bayesian optimization and Gaussian-process models can efficiently explore both atomic ordering and adsorption geometries, enabling the identification of stable catalyst structures in complex alloy systems.

After completing this tutorial we can:
* Set up and implement BEACON algorithm
* Generate and relax random alloy nanoparticle structures
* Identify the most stable atomic arrangements
* Perform adsorption site searches
